# 05. Attention và TabICLv2 (Bước 7)

**Nhiệm vụ của notebook này:** hiểu học trong ngữ cảnh, tính attention bằng tay, rồi chạy TabICLv2 thật nếu máy có internet (mục 2.7, 2.8).

Chạy từng ô từ trên xuống. Notebook tự đọc `data.csv` (Colab sẽ hỏi bạn tải lên; trên máy thì đặt tại `data/data.csv`).

Nên đọc theo thứ tự các notebook 01 → 06. Phần trước: 04.

In [1]:
# Ô thiết lập: cài thư viện (nếu thiếu), nạp thư viện và đọc dữ liệu
# Cài thư viện còn thiếu (tự động). Chạy được cả trên máy cá nhân lẫn Google Colab.
import importlib.util, subprocess, sys
_can = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "sklearn": "scikit-learn"}
_thieu = [g for m, g in _can.items() if importlib.util.find_spec(m) is None]
if _thieu:
    print("Máy chưa có:", ", ".join(_thieu), "-> đang cài, đợi một chút ...")
    if importlib.util.find_spec("pip") is None:      # môi trường ảo (venv) có thể chưa có pip
        import ensurepip; ensurepip.bootstrap()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_thieu])
    importlib.invalidate_caches()
    print("Cài xong.")

import io, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

warnings.filterwarnings("ignore")
SEED = 42                               # hạt giống ngẫu nhiên cố định: chạy lại ra đúng kết quả cũ
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def doc_du_lieu():
    """Tìm data.csv trên máy; nếu đang ở Colab thì xin người dùng tải lên."""
    for p in ["data/data.csv", "../data/data.csv", "../../data/data.csv", "data.csv"]:
        if os.path.exists(p):
            print("Đọc dữ liệu từ", p)
            return pd.read_csv(p, sep=";")
    try:
        from google.colab import files
        print("Hãy chọn tệp data.csv trên máy của bạn ...")
        up = files.upload()
        return pd.read_csv(io.BytesIO(list(up.values())[0]), sep=";")
    except ImportError:
        raise FileNotFoundError(f"Không thấy data.csv (thư mục hiện tại: {os.getcwd()}). Hãy đặt tệp vào thư mục data/ của dự án.")


raw = doc_du_lieu()
raw.columns = [c.strip() for c in raw.columns]      # bỏ khoảng trắng/tab thừa ở tên cột
print("Kích thước:", raw.shape)

# --- Chuẩn bị chung (lặp lại ở mỗi notebook để mỗi tệp chạy độc lập) ---
d = raw[raw.Target != "Enrolled"].reset_index(drop=True)
d["y"] = (d.Target == "Dropout").astype(int)          # 1 = bỏ học (nguy cơ), 0 = tốt nghiệp (an toàn)
ENTRY = ["Tuition fees up to date", "Debtor", "Scholarship holder", "Age at enrollment"]
YEAR1 = ["Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
         "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)"]
COMPACT = ENTRY + YEAR1
ADMISSION_ONLY = ["Admission grade", "Previous qualification (grade)", "Age at enrollment",
                  "Scholarship holder", "Debtor", "Tuition fees up to date", "Gender", "Application order"]
ALL = [c for c in d.columns if c not in ("Target", "y")]
y = d.y.values
train_idx, test_idx = train_test_split(np.arange(len(d)), test_size=0.2, stratify=d.y, random_state=SEED)
ytr, yte = y[train_idx], y[test_idx]


def chuan_bi(cols, chuan_hoa=True):
    """Ma trận X cho tập huấn luyện và kiểm tra; chuẩn hóa bằng trung bình, độ lệch chuẩn của TẬP HUẤN LUYỆN."""
    X = d[cols].values.astype(float)
    Xtr, Xte = X[train_idx], X[test_idx]
    if chuan_hoa:
        mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd == 0] = 1
        Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    return Xtr, Xte


def hoi_quy_sklearn(X, y):
    try:
        return LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
    except Exception:
        return LogisticRegression(C=np.inf, max_iter=5000).fit(X, y)


def do_metrics(y_that, p, nguong=0.5):
    yh = (p >= nguong).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_that, yh).ravel()
    return {"accuracy": accuracy_score(y_that, yh), "precision": precision_score(y_that, yh, zero_division=0),
            "recall": recall_score(y_that, yh), "f1": f1_score(y_that, yh), "auc": roc_auc_score(y_that, p),
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}


CAU_HINH = {"A. Lúc nhập học (8 cột)": ADMISSION_ONLY,
            "B1. Nhập học + hết HK1 (6 cột)": ENTRY + YEAR1[:2],
            "B. Nhập học + hết HK2 (8 cột)": COMPACT,
            "C. Đầy đủ 35 cột": ALL}


Hãy chọn tệp data.csv trên máy của bạn ...


Saving data.csv to data.csv
Kích thước: (4424, 37)


---
# Bước 7. TabICLv2: học trong ngữ cảnh và attention (mục 2.7 và 2.8 của đề)

## Vì sao cần bước này
Đề tài là **TabICLv2** (Qu, Holzmüller, Varoquaux, Le Morvan, ICML 2026), một mô hình nền cho dữ liệu dạng bảng. Điểm khác biệt lớn nhất so với Hồi quy Logistic:

> Hồi quy Logistic **học trên dữ liệu của mình** (Gradient Descent chạy trên sinh viên của ta). TabICLv2 **không huấn luyện lại**: nó đã được huấn luyện *một lần* ở chỗ khác, trên rất nhiều bảng dữ liệu *giả lập*. Khi dùng, ta chỉ đưa các sinh viên đã biết kết cục vào làm "ngữ cảnh", rồi nó đoán cho sinh viên mới **chỉ bằng một lượt tính**, không cập nhật trọng số nào. Cách này gọi là **học trong ngữ cảnh** (in-context learning).

## Attention từ con số 0
Hãy hình dung em mới cần đoán, và có một nhóm em cũ đã biết kết cục. Ta làm ba việc:
1. **So sánh**: đo xem em mới *giống* từng em cũ đến đâu (điểm số).
2. **Đổi thành trọng số**: dùng hàm **softmax** để biến các điểm số thành các trọng số dương, cộng lại bằng 1 (em càng giống thì trọng số càng lớn).
3. **Bỏ phiếu có trọng số**: xác suất bỏ học của em mới = tổng trọng số của những em cũ *đã bỏ học*.

Đó chính là attention. Trong mô hình thật, mỗi em được biến thành ba vector bằng ba ma trận trọng số *học được*: **truy vấn Q** ("tôi đang tìm ai giống tôi?"), **khóa K** ("tôi trông như thế nào?") và **giá trị V** ("tôi mang thông tin gì?"). Điểm số là tích vô hướng Q·K; công thức đầy đủ là softmax(QKᵀ/√d)·V. Trong TabICLv2, nhãn của em cũ được cộng thêm vào biểu diễn của em đó, nhờ vậy thông tin kết cục đi theo trọng số attention sang em mới.

Ta thử trước bằng **phiên bản đơn giản tự viết** (không có trọng số học được), rồi mới tới TabICLv2 thật.

In [2]:
def attention_don_gian(Xngucanh, y_ngucanh, Xhoi, tau):
    """Một đầu attention không có trọng số học được.
    Điểm = -(khoảng cách)^2 / (2*tau): càng giống, điểm càng cao. tau điều khiển độ rộng của sự chú ý."""
    d2 = ((Xhoi[:, None, :] - Xngucanh[None, :, :]) ** 2).sum(-1)
    diem = -d2 / (2 * tau)
    diem -= diem.max(1, keepdims=True)                        # trừ max cho ổn định số
    trong_so = np.exp(diem); trong_so /= trong_so.sum(1, keepdims=True)   # softmax
    return trong_so @ y_ngucanh                               # bỏ phiếu có trọng số


Xa, Xb = chuan_bi(COMPACT, True)
rng = np.random.RandomState(SEED)
hang = []
for n_nc in [50, 200, 800, len(ytr)]:
    idx = rng.choice(len(ytr), n_nc, replace=False) if n_nc < len(ytr) else np.arange(len(ytr))
    for tau in [0.05, 0.5, 8.0]:
        p = attention_don_gian(Xa[idx], ytr[idx].astype(float), Xb, tau)
        hang.append({"số em trong ngữ cảnh": n_nc, "tau": tau, **{k: v for k, v in do_metrics(yte, p).items()
                                                                    if k in ("accuracy", "recall", "auc")}})
bang_att = pd.DataFrame(hang)
display(bang_att.round(3))

,số em trong ngữ cảnh,tau,accuracy,recall,auc
0,50,0.05,0.860,0.845,0.874
1,50,0.50,0.868,0.799,0.901
2,50,8.00,0.708,0.261,0.927
3,200,0.05,0.854,0.842,0.882
4,200,0.50,0.869,0.806,0.911
5,200,8.00,0.767,0.475,0.933
6,800,0.05,0.871,0.813,0.889
7,800,0.50,0.879,0.782,0.931
8,800,8.00,0.752,0.437,0.929
9,2904,0.05,0.897,0.820,0.918


In [3]:
# Ví dụ tính tay: 1 em mới, 4 em trong ngữ cảnh, 2 chỉ số (số môn qua HK1 và HK2), dạng ĐÚNG như attention thật: Q·K rồi softmax
# Ma trận Q, K lấy bằng ma trận đơn vị (TỰ LẬP, để dễ nhìn; trong mô hình thật chúng được học). V chính là nhãn.
f2 = ["Curricular units 1st sem (approved)", "Curricular units 2nd sem (approved)"]
Xf = d[f2].values.astype(float)
mu2, sd2 = Xf[train_idx].mean(0), Xf[train_idx].std(0)
Z = (Xf - mu2) / sd2


def tim(pool, nhan, a, b_):
    return [i for i in pool if y[i] == nhan and Xf[i, 0] == a and Xf[i, 1] == b_][:1]


ngu_canh = tim(train_idx, 1, 0, 0) + tim(train_idx, 0, 6, 6) + tim(train_idx, 1, 2, 1) + tim(train_idx, 0, 5, 5)
em_moi = tim(test_idx, 1, 1, 1)[0]
K = Z[ngu_canh]                  # "khóa" của 4 em cũ
Q = Z[em_moi]                    # "truy vấn" của em mới
V = y[ngu_canh].astype(float)    # "giá trị": nhãn (1 = bỏ học)

diem = K @ Q / np.sqrt(2)        # tích vô hướng Q·K chia căn bậc hai của số chiều (d = 2)
tw = np.exp(diem - diem.max()); tw /= tw.sum()
ket_qua_tay = pd.DataFrame({"số môn qua (HK1; HK2)": [f"({int(Xf[i, 0])}; {int(Xf[i, 1])})" for i in ngu_canh],
                            "kết cục thật": ["bỏ học" if v else "an toàn" for v in V],
                            "điểm Q·K/√d": diem.round(2), "trọng số softmax": tw.round(3)})
display(ket_qua_tay)
print(f"Em mới qua ({int(Xf[em_moi, 0])}; {int(Xf[em_moi, 1])}) môn. P(bỏ học) = tổng trọng số × nhãn = {float(tw @ V):.3f}")
print("Kết cục thật của em mới:", "bỏ học" if y[em_moi] else "an toàn")

,số môn qua (HK1; HK2),kết cục thật,điểm Q·K/√d,trọng số softmax
0,(0; 0),bỏ học,2.42,0.633
1,(6; 6),an toàn,-0.67,0.029
2,(2; 1),bỏ học,1.64,0.291
3,(5; 5),an toàn,-0.16,0.048


Em mới qua (1; 1) môn. P(bỏ học) = tổng trọng số × nhãn = 0.923
Kết cục thật của em mới: bỏ học


## Đọc kết quả của attention đơn giản
- **Ngữ cảnh càng nhiều thì càng tốt** (với `tau` vừa phải như 0,5, cột `auc` tăng từ 0,90 lên 0,95 khi ngữ cảnh tăng từ 50 đến toàn bộ 2.904 em), vì có thêm nhiều em giống để tham khảo.
- **Tham số `tau` điều khiển mức tập trung.** `tau` rất nhỏ: chỉ nhìn vài em giống nhất, dễ bị nhiễu cục bộ (giống nhớ vặt). `tau` rất lớn (8,0): nhìn mọi em gần như đều nhau, mô hình thành dự đoán theo trung bình chung và **bỏ sót nhiều em bỏ học** (Recall tụt thấp). Đây là ví dụ trực quan cho thế cân bằng giữa overfitting và underfitting.
- **Ví dụ tính tay:** em mới có hồ sơ gần nhất với hai em cũ đều đã bỏ học nên nhận gần hết trọng số, và dự đoán P(bỏ học) cao. Đúng với kết cục thật.

⚠️ Đây **chỉ là bản minh họa cùng ý tưởng**, không có trọng số học được và không phải TabICLv2. Không dùng nó để suy ra chất lượng của TabICLv2.

## TabICLv2 thật, theo bài gốc
Bốn điều đáng nhớ (tóm tắt từ bài, cần đối chiếu khi nộp):
1. **Ba tầng nối nhau.** (i) *Nhúng theo cột*: mỗi chỉ số của một em thành vector (bằng set transformer); (ii) *Tương tác theo hàng*: gộp các chỉ số của một em thành một vector đại diện (dùng token [CLS]); (iii) *Học trong ngữ cảnh theo tập dữ liệu*: các em cần đoán "chú ý" vào các em đã có nhãn (bài nói rõ: các em kiểm tra chỉ chú ý vào các em huấn luyện).
2. **QASSMax.** Softmax thường bị "loãng" khi ngữ cảnh dài: mẫu số lớn lên nên sự chú ý dàn đều. QASSMax co giãn truy vấn theo log của số mẫu, kèm một cổng phụ thuộc truy vấn, để vẫn tập trung được khi ngữ cảnh dài.
3. **Huấn luyện trước một lần trên dữ liệu giả lập.** Bảng giả lập sinh từ các mô hình nhân quả ngẫu nhiên (đồ thị có hướng ngẫu nhiên, nhiều loại hàm: mạng nơ-ron, cây, quá trình Gauss...). Mục tiêu học là cross-entropy: đoán nhãn các hàng kiểm tra khi đã cho xem các hàng huấn luyện. Bài cho rằng cách này tương ứng với một phân phối tiên nghiệm (prior) ngầm trên các bộ dữ liệu. Mô hình khoảng 28 triệu tham số, huấn luyện qua ba giai đoạn với kích thước dữ liệu tăng dần.
4. **Bộ tối ưu Muon** thay cho AdamW. Muon làm trực giao hóa cập nhật gradient bằng phép lặp Newton–Schulz (bài mô tả là dốc nhất theo chuẩn phổ) và cho phép dùng tốc độ học cao hơn. Đây là chuyện của giai đoạn huấn luyện trước, nên **ta không thể chỉnh nó trên dữ liệu sinh viên**.

In [4]:
# Chạy TabICLv2 thật (CẦN INTERNET để tải trọng số từ HuggingFace; trên Colab chạy được, có GPU thì nhanh hơn)
RUN_TABICL = True          # đổi thành False nếu muốn bỏ qua ô này
ket_qua_tabicl = None

if RUN_TABICL:
    try:
        try:
            from tabicl import TabICLClassifier
        except ImportError:
            # Không tự cài: tabicl kéo theo PyTorch (vài GB). Muốn chạy thì cài một lần: %pip install tabicl
            raise ImportError("chưa cài tabicl (và PyTorch). Chạy ô '%pip install tabicl' rồi chạy lại ô này, hoặc đặt RUN_TABICL = False")
        import time
        hang = []
        for ten_ch, cot in CAU_HINH.items():
            Xa_raw, Xb_raw = chuan_bi(cot, chuan_hoa=False)       # TabICL tự xử lý thang đo, đưa dữ liệu thô vào
            t0 = time.time()
            clf = TabICLClassifier(n_estimators=8, random_state=SEED)
            clf.fit(Xa_raw, ytr)                                    # "fit" chỉ là đưa ngữ cảnh vào, KHÔNG cập nhật trọng số
            p = clf.predict_proba(Xb_raw)[:, 1]
            hang.append({"cấu hình": ten_ch, **do_metrics(yte, p), "giây": round(time.time() - t0, 1)})
            print("xong:", ten_ch)
        ket_qua_tabicl = pd.DataFrame(hang)
        display(ket_qua_tabicl[["cấu hình", "accuracy", "precision", "recall", "f1", "auc", "giây"]].round(3))
    except Exception as e:
        print("Chưa chạy được TabICLv2 ở môi trường này.")
        print("Lý do:", repr(e)[:300])
        print("Thường do không tải được trọng số (không có internet hoặc bị chặn HuggingFace). Trên Colab thì chạy được.")

INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt: reconstructing file:   0%|          |  0.00B /  110MB            

tabicl-classifier-v2-20260212.ckpt: downloading bytes:           |  0.00B            

xong: A. Lúc nhập học (8 cột)
xong: B1. Nhập học + hết HK1 (6 cột)
xong: B. Nhập học + hết HK2 (8 cột)
xong: C. Đầy đủ 35 cột


,cấu hình,accuracy,precision,recall,f1,auc,giây
0,A. Lúc nhập học (8 cột),0.756,0.715,0.627,0.668,0.823,7.0
1,B1. Nhập học + hết HK1 (6 cột),0.879,0.863,0.820,0.841,0.934,1.2
2,B. Nhập học + hết HK2 (8 cột),0.913,0.908,0.866,0.886,0.955,1.1
3,C. Đầy đủ 35 cột,0.946,0.942,0.919,0.930,0.977,1.9


## Thầy có thể hỏi
- *Học trong ngữ cảnh khác huấn luyện thường ở đâu?* Huấn luyện thường cập nhật trọng số trên dữ liệu của mình. Học trong ngữ cảnh giữ nguyên trọng số: các ví dụ có nhãn được đưa vào làm ngữ cảnh và mô hình suy luận ngay trong một lượt tính.
- *Attention làm gì?* So sánh em mới với các em cũ, đổi độ giống thành trọng số bằng softmax, rồi bỏ phiếu có trọng số theo nhãn.
- *Q, K, V là gì?* Truy vấn, khóa, giá trị: ba vector sinh ra từ mỗi em bằng ba ma trận trọng số học được.
- *Muon khác AdamW thế nào?* Muon trực trục hóa cập nhật gradient của các ma trận trọng số (Newton–Schulz), cho phép tốc độ học cao hơn. Chỉ dùng lúc huấn luyện trước.
- *Sao không đổi tốc độ học của Muon trong thí nghiệm?* Vì TabICLv2 không được huấn luyện lại trên dữ liệu sinh viên; thí nghiệm tối ưu phải làm trên mô hình đối chứng.